## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

In [1]:
!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

## 1. Import Libraries

In [2]:
import os
import re
import gc
import math
import json
import random
import unicodedata
import fitz  # PyMuPDF
import torch
import inspect
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, set_seed,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)
from google.colab import userdata
from huggingface_hub import HfApi

In [3]:
WRITE_TOKEN = userdata.get('HF_TOKEN_WRITE')

api = HfApi(token=WRITE_TOKEN)

## 2. Global Configurations

In [4]:
@dataclass
class Config:
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Output directories.
    output_dir: str = "/content/pharma_tinyllama_lora_output"
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"
    processed_data_dir: str = "/content/pharma_processed_data"

    # Text preprocessing.
    min_chars_per_paragraph: int = 80
    block_size: int = 512

    # Train/eval split.
    test_size: float = 0.15
    seed: int = 42

    # LoRA parameters.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training parameters.
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 10
    save_steps: int = 25
    save_total_limit: int = 2

    # For a quick demo, set max_steps to 20 or 30.
    # For a complete run, keep it as -1.
    max_steps: int = -1

In [5]:
config = Config()
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, logging_steps=5, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [6]:
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [7]:
config.processed_data_dir

'/content/pharma_processed_data'

In [8]:
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [9]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

## 3. Check PDF

In [10]:
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


## 4. Extract Text from PDF

In [11]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [12]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge.pdf'

In [13]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [14]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [15]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


## 5. Text Cleaning Utilities

In [16]:
def normalize_unicode(text: str) -> str:
    # Normalize Unicode characters to reduce hidden character issues.
    return unicodedata.normalize("NFKC", text)

In [17]:
def clean_pdf_text(text: str) -> str:
    # Clean raw PDF text while preserving meaningful scientific content.
    text = normalize_unicode(text)

    # This removes special hidden characters.
    # \u200b
    # Is a zero-width space. It is invisible, but it exists inside the text.
    # \ufeff
    # Is a Byte Order Mark / hidden marker. It sometimes appears during PDF/text extraction.
    # It is important to remove these, otherwise the tokenizer can get confused.
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # Fix hyphenated line breaks: for example, "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Convert multiple spaces/tabs into a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize repeated blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove standalone page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Preserve paragraph boundaries, but remove line wrapping inside paragraphs.
    paragraphs = re.split(r"\n\s*\n", text)

    cleaned_paragraphs = []

    for paragraph in paragraphs:
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            cleaned_paragraphs.append(paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [20]:
# pdf_pages

In [21]:
cleaned_pages = []

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [23]:
# cleaned_pages

In [24]:
print("Cleaned page preview:\n")
print(cleaned_pages[0]["text"])

Cleaned page preview:

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenou

## 6. Splitting Cleaned Pages into Paragraph

In [25]:
def split_into_paragraph_records(cleaned_pages: List[Dict[str, Any]], min_chars: int = 80) -> List[Dict[str, Any]]:
    records = []
    seen = set()

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])
        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()
            if len(paragraph) < min_chars:
                continue

            # Deduplicate exact repeated paragraphs.
            normalized_key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if normalized_key in seen:
                continue
            seen.add(normalized_key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return records

In [26]:
config.min_chars_per_paragraph

80

In [27]:
paragraph_records = split_into_paragraph_records(
    cleaned_pages,
    min_chars=config.min_chars_per_paragraph
)

In [28]:
print(f"Total paragraph records: {len(paragraph_records)}")
for i, record in enumerate(paragraph_records[:5]):
    print("=" * 100)
    print(f"Record {i} | Page {record['source_page']} | Characters: {record['char_count']}")
    print(record["text"])

Total paragraph records: 9
Record 0 | Page 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Record 1 | Page 1 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, w

In [30]:
# paragraph_records

In [31]:
config.processed_data_dir

'/content/pharma_processed_data'

In [32]:
config.processed_data_dir

'/content/pharma_processed_data'

## 7. Save extracted and cleaned corpus for auditability

In [33]:
raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


## 8. Create Hugging Face Dataset

In [34]:
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

text_dataset = Dataset.from_list(paragraph_records)

In [35]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [36]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


## 9. Train / Eval Split

In [37]:
split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## 10. Load Tokenizer

In [38]:
config.model_name

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [39]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

#### Some Llama-style models do not define a pad token.
#### For causal LM fine-tuning, using EOS as PAD is a common practical choice.

In [40]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


## 11. Tokenization and text packing

In [41]:
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [42]:
def group_texts(examples):
    # Concatenate tokenized texts and split into fixed-length blocks.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])

    # Drop the small remainder so all blocks have exactly block_size tokens.
    total_length = (total_length // config.block_size) * config.block_size

    if total_length == 0:
        return {k: [] for k in concatenated_examples.keys()}

    result = {
        k: [t[i : i + config.block_size] for i in range(0, total_length, config.block_size)]
        for k, t in concatenated_examples.items()
    }

    # For causal LM, labels are the same as input_ids.
    # The model internally shifts labels for next-token prediction.
    result["labels"] = result["input_ids"].copy()
    return result

In [43]:
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

In [44]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

In [45]:
final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing tokenized text into blocks of {config.block_size}",
)

final_datasets

Packing tokenized text into blocks of 512:   0%|          | 0/7 [00:00<?, ? examples/s]

Packing tokenized text into blocks of 512:   0%|          | 0/2 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 6
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 0
    })
})

In [46]:
if len(final_datasets["train"]) == 0:
    raise ValueError(
        "No training blocks were created. Reduce config.block_size or provide a larger PDF corpus."
    )

In [47]:
sample = final_datasets["train"][0]

In [49]:
# sample

In [50]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

## 12. Load model with QLoRA-friendly configuration

We use 4-bit quantized loading when CUDA is available.

Why?

- Lower GPU memory usage
- Faster experimentation
- Practical for Colab-style environments
- Common industry approach for parameter-efficient fine-tuning

If CUDA is not available, the notebook falls back to normal loading, but training will be slow on CPU.

In [51]:
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [52]:
# Clear memory before loading the model.
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Required for stable training with k-bit quantized models.
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to avoid warnings and reduce memory usage.
base_model.config.use_cache = False

print("Base model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model loaded successfully.


## 13. Apply LoRA adapters

- LoRA trains a small number of adapter parameters instead of updating all base model weights.
- This is cheaper than full fine-tuning and is widely used in real projects.

In [53]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [54]:
model = get_peft_model(base_model, lora_config)

In [55]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?

Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

## 14. Data collator

In [56]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 15. Training arguments

In [57]:
eval_mode = "steps" if len(final_datasets["validation"]) > 0 else "no"

In [58]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,

    # Use warmup_steps instead of deprecated warmup_ratio.
    # For small demo training, 5-10 warmup steps are enough.
    warmup_steps=5,

    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [59]:
training_args_params = inspect.signature(TrainingArguments.__init__).parameters

In [60]:
if "eval_strategy" in training_args_params:
    training_kwargs["eval_strategy"] = eval_mode
elif "evaluation_strategy" in training_args_params:
    training_kwargs["evaluation_strategy"] = eval_mode

In [61]:
safe_training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in training_args_params
}

In [62]:
safe_training_kwargs

{'output_dir': '/content/pharma_tinyllama_lora_output',
 'num_train_epochs': 3.0,
 'max_steps': -1,
 'per_device_train_batch_size': 1,
 'per_device_eval_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'learning_rate': 0.0002,
 'warmup_steps': 5,
 'weight_decay': 0.01,
 'logging_steps': 5,
 'eval_steps': 10,
 'save_steps': 25,
 'save_total_limit': 2,
 'fp16': True,
 'bf16': False,
 'report_to': 'none',
 'remove_unused_columns': False,
 'eval_strategy': 'no'}

In [63]:
removed_args = set(training_kwargs.keys()) - set(safe_training_kwargs.keys())
if removed_args:
    print("Removed unsupported TrainingArguments:", removed_args)

In [64]:
training_args = TrainingArguments(**safe_training_kwargs)

In [66]:
# training_args

## 16. Build Trainer

In [67]:
trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"] if len(final_datasets["validation"]) > 0 else None,
    data_collator=data_collator,
)

In [68]:
trainer_params = inspect.signature(Trainer.__init__).parameters

In [69]:
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

In [70]:
safe_trainer_kwargs = {
    key: value
    for key, value in trainer_kwargs.items()
    if key in trainer_params
}

In [71]:
removed_trainer_args = set(trainer_kwargs.keys()) - set(safe_trainer_kwargs.keys())
if removed_trainer_args:
    print("Removed unsupported Trainer arguments:", removed_trainer_args)

In [72]:
trainer = Trainer(**safe_trainer_kwargs)

print("Trainer is ready.")

Trainer is ready.


## 17. Start Training

In [73]:
train_result = trainer.train()

print("Training completed.")
print(train_result)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


Training completed.
TrainOutput(global_step=3, training_loss=2.1406920750935874, metrics={'train_runtime': 12.1518, 'train_samples_per_second': 1.481, 'train_steps_per_second': 0.247, 'total_flos': 57901993426944.0, 'train_loss': 2.1406920750935874, 'epoch': 3.0})


## 18. Save Adapter and Tokenizer

In [74]:
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['adapter_model.safetensors', 'tokenizer_config.json', 'adapter_config.json', 'README.md', 'tokenizer.json']


## 19. Push LoRA adapter to Hugging Face Hub

In [80]:
repo_id = "saadtariq/non-instruction-pharma-finetuning"

model.push_to_hub(
    repo_id,
    private=True,
    token=WRITE_TOKEN
)

tokenizer.push_to_hub(
    repo_id,
    private=True,
    token=WRITE_TOKEN
)

print(f"LoRA adapter pushed to: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  16%|#5        | 7.93MB / 50.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.


LoRA adapter pushed to: https://huggingface.co/saadtariq/non-instruction-pharma-finetuning


## 20. Reload base model + LoRA adapter correctly

In [81]:
# Clean old objects to free memory.
del trainer
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [82]:
if use_cuda:
    reload_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=reload_bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)
inference_model.eval()

print("Base model + LoRA adapter loaded successfully for inference.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Base model + LoRA adapter loaded successfully for inference.


## 21. Inference helper

In [83]:
def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

## 22. Test text continuation

In [84]:
prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could help explain the mechanism of action of Metformin.
Metformin is an orally available compound that is used to treat

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

## 23. Optional merge step

In [85]:
merged_model_dir = "/content/pharma_tinyllama_merged_model"
merged_model = inference_model.merge_and_unload()
merged_model.save_pretrained(merged_model_dir)
inference_tokenizer.save_pretrained(merged_model_dir)
print(f"Merged model saved to: {merged_model_dir}")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model
